# Entrega 2 visualización de datos
### Aaron Vargas

# Carga de datos

In [20]:
import pandas as pd

In [21]:
csv = './respuestas.csv'

In [22]:
df = pd.read_csv(csv, delimiter=';')

In [23]:
df.head()

,¿Cuál de los siguientes sectores de la sociedad consideras que es más afectado actualmente por el calentamiento global?,"Si tuvieras la oportunidad de migrar a otro país exclusivamente en búsqueda de un clima más de tu gusto, ¿cuál elegirías?","En retrospectiva a tu infancia, ¿cómo percibes que ha cambiado la temperatura promedio en la ciudad donde creciste?","En una escala del 1 al 10, ¿Qué grado de responsabilidad sientes frente al fenómeno del calentamiento global?",Elige 4 transportes que usas frecuentemente,¿Qué acciones concretas realizas para contribuir a la prevención del cambio climático?
0,Ecosistemas Marinos,Nueva Zelanda,Ligeramente más cálida,5,"Bus Eléctrico (E01, E02, E03);Micro;A Pie","Tomar duchas cortas, no prender luces de día, ..."
1,Agricultura,Noruega,Significativamente más cálida,3,"Bus Eléctrico (E01, E02, E03);Metro;Micro",Ninguna
2,Agricultura,suiza,Ligeramente más cálida,3,Metro;Auto Personal,Ninguna
3,Ecosistemas Marinos,Suecia,Significativamente más cálida,3,Metro;Auto Personal;Micro;A Pie,Me preocupo de no botar basura donde no corres...
4,Ecosistemas Marinos,"Me quedaría en el centro de Chile, posee un cl...",Ligeramente más cálida,5,A Pie,Disminuir el uso de autos cuando no es estrict...


In [24]:
pregunta1= df.columns[0]
pregunta2= df.columns[1]
pregunta3= df.columns[2]
pregunta4= df.columns[3]
pregunta5= df.columns[4]
pregunta6= df.columns[5]

# Limpieza de datos

In [25]:
print(list(df.loc[:,pregunta2]))

['Nueva Zelanda', 'Noruega', 'suiza', 'Suecia', 'Me quedaría en el centro de Chile, posee un clima templado que es de mi gusto.', 'Canada', 'Alemania', 'Suecia', 'Un lugar en donde siempre haga calor', 'Suecia', 'Holanda', 'Canada', 'australia', 'Como clima Mexico', 'Nueva Zelanda', 'Colombia', 'Ninguno', 'Irlanda o escocia', 'Canada', 'Japon', 'Puerto Rico']


In [26]:
# se abrevian las respuestas largas a 1 solo país concreto
df.loc[4, pregunta2] = 'Chile'
df.loc[8, pregunta2] = 'lugar_caluroso'
df.loc[13, pregunta2] = 'Mexico'
df.loc[:, pregunta2] = df.loc[:, pregunta2].apply(lambda x: x.capitalize())

# creación de features

In [27]:
# funcion para agrupar la responsabilidad climatica de valores continuos en clases categoricas
def responsabilidad(x):
    x = int(x)

    if x <= 3:
        return 'Responsabilidad baja'

    elif x <= 7:
        return 'Responsabilidad media'

    else:
        return 'Responsabilidad alta'

In [28]:
respn = 'responsabilidad_cat'
df[respn] = df[pregunta4].apply(responsabilidad)

In [29]:
# se asigna id a cada tipo de respuestas para realizar los links
source='id_temp'
target= 'id_country'
target2 = 'id_respn'
df[source] = pd.factorize(df[pregunta3])[0]
df[target] = pd.factorize(df[pregunta2])[0] + df[source].max() + 1
df[target2] = pd.factorize(df[respn])[0] + df[target].max() + 1
print(df.shape)

(21, 10)


In [30]:
# respuestas de percepcion de temperatura y de los paises
labels = pd.concat([df[pregunta3], df[pregunta2], df[respn]]).unique() # asigna por orden de aparicion, no sera necesario ordenar labels vs target,source
print(labels.shape)
labels = list(labels)

(23,)


In [31]:
# se calcula los values para el grosor de las lineas
values='values'

# flow de la percepción de temperatura  apreferencia de paises
flows = (
    df.groupby([source, target, pregunta3, pregunta2])
    .size()
    .reset_index(name=values)
)

In [32]:
# flujo de la preferencia a los paises a el sentido de la responsabilidad
flow2 = (
    df.groupby([target, target2 , pregunta2, respn])
    .size()
    .reset_index(name=values)
)

In [33]:
# flujo de responsabilidad a tipo de transporte
target3='transporte_id'
df[pregunta5] = df[pregunta5].str.split(';') # se crea la lista del string
df_t = df.explode(pregunta5) #se expanden las instancias por cada elemento de la lista
df_t[target3] = pd.factorize(df_t[pregunta5])[0] + df[target2].max() + 1 # s ele da un id a cada tipo de transporte

flow3 = (
    df_t.groupby([target2, target3, pregunta5,])
    .size()
    .reset_index(name=values)
)

min = flow3[values].min()
max = flow3[values].max()
flow3[values] = flow3[values].apply(lambda x : 5* (x - min +1)/(max-min +1) )

labels = labels + list(flow3[pregunta5].unique())
labels = [x if x!= 'Bus Eléctrico (E01, E02, E03)' else 'Bus electrico' for x in labels ]


In [34]:
# colores percepcion calorica catgorico
colores = [
    '#dc4646',
    '#b45078',
    '#825aaa',
    '#466edc',
]
# colores para paises de preferencia categorico
colores = colores + ['#80bc96']*df[pregunta2].unique().shape[0]

# colores de nivel de sentido de la responsabilidad categorico
colores = colores + [
    '#be5050',
    '#dcb45a',
    '#50aa78'
]

# colores de tipo de transporte
colores = colores + [
    '#D7C148',  # amarillo mostaza
    '#826E00',  # oliva oscuro

    '#3FA7D6',  # celeste fuerte
    '#1F4E79',  # azul petróleo

    '#E07A5F',  # coral
    '#7B2D26',  # rojo arcilla oscuro

    '#7BC950',  # verde lima natural
    '#5A189A',  # violeta profundo
]


len(colores)

31

# Visualización

In [35]:
from plotly.graph_objects import Figure, Sankey

In [36]:
sources = list(flows[source])
sources = sources + list(flow2[target])
sources = sources + list(flow3[target2])

targets = list(flows[target])
targets = targets + list(flow2[target2])
targets = targets + list(flow3[target3])

valores = list(flows[values])
valores = valores + list(flow2[values])
valores = valores + list(flow3[values])

In [37]:
nodes = dict(
    pad=15,
    thickness=15,
    line = dict(color = "black", width = 0.5),
    label = labels ,
    color = colores,
)

links = dict(
    source = sources,
    target = targets,
    value = valores
)

data = Sankey(
    node = nodes,
    link = links
)

fig = Figure(data=[data])
fig.update_layout(title_text="Flujo de percepción vs preferencia de pais de vivencia vs sentido de la responsabilidad", font_size=15)
fig.show()